<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="20%">
</div>

<br>

# TITANIC SURVIVOR ANALYSIS

<br>

**About:** A complete end-to-end machine learning pipeline on the Titanic passenger dataset, covering exploratory data analysis, feature engineering, SQLite storage, and multi-model comparison - the same workflow you will encounter in data science interviews and on the job.

**Learning Goals:**
1. Load, inspect, and describe a tabular dataset using pandas
2. Identify and handle missing values using domain-informed imputation strategies
3. Engineer new features from raw columns (titles from names, family size, fare quartiles)
4. Encode categorical variables as numeric representations for use in scikit-learn
5. Store a preprocessed dataset in SQLite and reload it for modeling
6. Train and compare five classifiers: Logistic Regression, KNN, SVM, XGBoost, and Random Forest
7. Interpret feature importance scores from a Random Forest model
8. Explain why a model's accuracy ceiling reflects real-world randomness, not a technical failure

**Keywords:** titanic, pandas, feature engineering, sqlite, classification, scikit-learn, xgboost, eda

**Prerequisite Knowledge:** (1) Python basics (variables, loops, functions), (2) pandas fundamentals (read_csv, head, describe, groupby), (3) basic probability (what a proportion means)

**Target User:** Data science job seekers or students who want to practice a full end-to-end ML pipeline interview on a well-understood dataset

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: IMPORTS AND SETUP](#Part_0)
> #### [PART 1: LOADING THE DATA](#Part_1)
> #### [PART 2: EXPLORATORY DATA ANALYSIS](#Part_2)
> #### [PART 3: FEATURE ENGINEERING AND PREPROCESSING](#Part_3)
> #### [PART 4: STORING PREPROCESSED DATA IN SQLITE](#Part_4)
> #### [PART 5: MODEL BUILDING AND EVALUATION](#Part_5)

#### APPENDIX

> #### [APPENDIX I: WHY DO MODELS MAX OUT AT ~80%?](#Appendix_1)
> #### [APPENDIX II: RESOURCES AND REFERENCES](#Appendix_2)

<br>

<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **IMPORTS** AND SETUP

___

**Note:** This notebook requires the `xgboost` package. If you do not have it installed, run the following in your terminal before launching Jupyter:

```
conda install py-xgboost
```

Or install directly from a notebook cell - remove the `#` in the cell below to uncomment.

___

In [ ]:
#!pip install py-xgboost

In [ ]:
# Importing essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Set plot style and options for display
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
pd.set_option('display.max_columns', 100)

# Filter warnings
import warnings
warnings.filterwarnings('ignore')

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **LOADING** THE DATA

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/news_titanic.jpg" align="center" width="50%" padding="10px"><br>
    <br>
    April 15, 1912 - the morning after
</div>

The sinking of the RMS Titanic is one of the most infamous shipwrecks in history. On April 15, 1912, during her maiden voyage, the Titanic sank after colliding with an iceberg, resulting in the loss of 1,502 lives out of 2,224 passengers and crew. Survival was not random - it was shaped by social class, gender, age, and chance.

In this notebook, we explore passenger data, build hypotheses about what predicted survival, and train classification models to predict survival outcomes - practicing the full data science interview workflow end-to-end on a dataset small enough to understand in full.

In [ ]:
# Load the Titanic dataset from CSV files
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

# Combine train and test datasets for preprocessing
combine = [train_df, test_df]

# Preview the first few rows of the training dataset
train_df.head()

<br>

**Supporting function: distribution plot**

In [ ]:
# Special distribution plot (will be used later)
def plot_distribution( df , var , target , **kwargs ):
    row = kwargs.get( 'row' , None )
    col = kwargs.get( 'col' , None )
    facet = sns.FacetGrid( df , hue=target , aspect=4 , row = row , col = col )
    facet.map( sns.kdeplot , var , shade= True )
    facet.set( xlim=( 0 , df[ var ].max() ) )
    facet.add_legend()
    plt.tight_layout()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **After loading a new dataset, what three things should you always check before touching the data? Write code to answer all three for `train_df`.**

<br>

<hr style="border: 2px solid#003262;" />

In [ ]:
# Check 1: shape (how many rows and columns?)
# TODO: your code here

# Check 2: column names and data types
# TODO: your code here

# Check 3: count of missing values per column
# TODO: your code here

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **EXPLORATORY** DATA ANALYSIS

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/Titanic_Variable.png" align="center" width="60%" padding="10px"><br>
    <br>
    Feature descriptions for the Titanic passenger manifest
</div>

In [ ]:
# Features/Variable names
train_df.columns

In [ ]:
# preview the data
train_df.head(5)

In [ ]:
# General data statistics
train_df.describe()

In [ ]:
# Data Frame information (null, data type etc)
train_df.info()

<br>

**Visualize feature distributions**

In [ ]:
# visualize feature distributions
train_df[
    ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
    ].hist(figsize=(13,10))
plt.show()

<br>

**Visualize response / target variable**

In [ ]:
# Balanced data set?

# Visualize the survivor vs non-survivor counts
sns.countplot(x='Survived', data=train_df)
plt.title('Survival Count on the Titanic')
plt.show()

# print counts
target_count = train_df['Survived'].value_counts()
target_count

___

**Note:** If the goal is prediction, unbalanced data introduce bias into a model. Balanced data are good for classification, but you lose information such as appearance frequencies - which may affect accuracy metrics themselves as well as production performance.

___

<br>

**Establish baseline prediction**

Any model that cannot beat the baseline is worse than always predicting the majority class.

In [ ]:
# Response variable counts
target_count

In [ ]:
# What is base line for prediction accuracy?
print("(survived = 0) --> ", target_count[0]/(sum(target_count)))

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Compute the survival rate broken down by both `Pclass` and `Sex` together (not separately). Which group had the highest survival rate? Which had the lowest? Does this suggest the two features carry overlapping or independent information?**

<br>

<hr style="border: 2px solid#003262;" />

In [ ]:
survival_by_class_sex = train_df.groupby(['Pclass', 'Sex'])['Survived'].mean().reset_index()
# Display sorted from highest to lowest survival rate
survival_by_class_sex.sort_values('Survived', ascending=False)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **FEATURE ENGINEERING** AND PREPROCESSING

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/titanic_voyage_map.png" align="center" width="50%" padding="10px"><br>
    <br>
    The Titanic's route from Southampton to its final position
</div>

**Brief remarks on the raw features:**

- `PassengerId` is an incrementing index with no predictive value.
- `Survived`, `Pclass`, `Age`, `SibSp`, `Parch`, and `Fare` are numeric (no encoding needed - but binning may help).
- `Sex` and `Embarked` are unordered categoricals that need encoding.
- `Name`, `Ticket`, and `Cabin` may contain useful signal but are high-cardinality or too sparse to use directly.

<hr style="border: 1px solid#003262;" />

**Dropping redundant features**

___

**Note:** Removing variables that convey information already captured by another feature reduces correlation between inputs and lowers the risk of overfitting.

___

In [ ]:
# Check dimensions of the train and test datasets
print("Shapes Before: (train) (test) = ", \
      train_df.shape, test_df.shape)

In [ ]:
# Drop the column 'PassengerID', need to do it for both test and training
train_df = train_df.drop(['PassengerId'], axis=1)
combine = [train_df, test_df]

print("Shapes After: (train) (test) =", train_df.shape, test_df.shape)

In [ ]:
# Check if there are null values in the datasets
print(train_df.isnull().sum())
print()
print(test_df.isnull().sum())


<br>

**Handling missing values**

In [ ]:
# Fill missing Age values with median age based on Pclass and Sex
for dataset in combine:
    dataset['Age'].fillna(dataset['Age'].median(), inplace=True)

# Fill missing Embarked values with the mode
train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)


___

**Exploration question:** Does a passenger's `Title` carry information not already captured by `Sex` and `Pclass`? The code below extracts titles from the `Name` column and checks.

___

<hr style="border: 1px solid#003262;" />

**Feature engineering: extracting titles from names**

A passenger's title (Mr, Mrs, Miss, Master, Rare) encodes sex and social status in a compressed signal. It also helps impute age - a "Master" is almost always a child.

In [ ]:
# Extract titles from the Name column
for dataset in combine:
    dataset['Title'] = dataset['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Visualize the survival rates by Title
sns.countplot(x='Title', hue='Survived', data=train_df)
plt.title('Survival Rate by Title')
plt.xticks(rotation=45)
plt.show()

___

**Note:** _Jonkheer_ was a Dutch noble title given to young, unmarried children of high-ranking knights or noblemen - considered the lowest rank of Dutch nobility. It was held by a single first-class passenger aboard the Titanic.

___

<br>

**Grouping rare titles into a single category**

In [ ]:
# Group rare titles together as 'Rare'
for dataset in combine:
    dataset['Title'] = dataset['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer'], 'Rare')


In [ ]:
# Visualize the survival rates by Title
sns.countplot(x='Title', hue='Survived', data=train_df)
plt.title('Survival Rate by Title')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Double check that our titles makes sense (by comparing to sex)
pd.crosstab(train_df['Title'], train_df['Sex'])

<hr style="border: 1px solid#003262;" />

**Mapping categorical variables to numeric**

<br>

**Normalizing to standard female titles**

In [ ]:
# We see common titles like Miss, Mrs, Mr, Master are dominant, we will
# correct some Titles to standard forms

for dataset in combine:
    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss') #Mademoiselle
    dataset['Title'] = dataset['Title'].replace('Ms', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme', 'Mrs') #Madame

In [ ]:
# We now have more logical (contemporary) titles, and fewer groups

train_df[['Title', 'Survived']].groupby(['Title']).mean()

<br>

**Survival count by title**

In [ ]:
# We can plot the survival chance for each title

sns.countplot(x='Survived', hue="Title", data=train_df, order=[1,0])
plt.xticks(range(2),['Made it','Deceased']);

In [ ]:
# Title dummy mapping: Map titles to binary dummy columns

for dataset in combine:
    binary_encoded = pd.get_dummies(dataset.Title)
    newcols = binary_encoded.columns
    dataset[newcols] = binary_encoded

train_df.head()

In [ ]:
# Remove unique variables for analysis (Title is generally bound to Name, so it's also dropped)

train_df = train_df.drop(['Name', 'Title'], axis=1)
test_df = test_df.drop(['Name', 'Title'], axis=1)
combine = [train_df, test_df]

In [ ]:
# sanity check

train_df.head()

<br>

**Mapping `Sex` to binary (male = 0, female = 1)**

In [ ]:
# convert categorical variable to numeric
for dataset in combine:
    dataset['Sex'] = dataset['Sex']. \
        map({'female': 1, 'male': 0}).astype(int)

train_df.head()

___

**Exploration question:** Are `Wealth` (Pclass) and `Sex` both predictors for survival? What if they are highly correlated? The imputation below uses both features together to produce more realistic age estimates than a global median would.

___

<br>

**Imputing missing age values using sex and passenger class**

Using a single global median would ignore the strong correlation between wealth, sex, and age. Stratified imputation produces more realistic estimates - a "Master" in third class is unlikely to be the same age as a "Mrs" in first class.

In [ ]:
# create empty array  for later use

guess_ages = np.zeros((2,3),dtype=int) 
guess_ages

In [ ]:
# Fill the NA's for the Age columns
# with "qualified guesses"

for idx,dataset in enumerate(combine):  
    # method adds a counter to an iterable and returns it in a form of enumerate object.     
    if idx==0:
        print('Working on Training Data set\n')
    else:
        print('-'*35)
        print('Working on Test Data set\n')
    
    print('Guess values of age based on sex and pclass of the passenger...')
    for i in range(0, 2):
        for j in range(0,3):
            guess_df = dataset[(dataset['Sex'] == i) \
                        &(dataset['Pclass'] == j+1)]['Age'].dropna()

            # Extract the median age for this group
            # (less sensitive) to outliers
            age_guess = guess_df.median()
          
            # Convert random age float to int
            guess_ages[i,j] = int(age_guess)
    
            
    print('Guess_Age table:\n',guess_ages)
    print ('\nAssigning age values to NAN age values in the dataset...')
    
    for i in range(0, 2):
        for j in range(0, 3):
            dataset.loc[ (dataset.Age.isnull()) & (dataset.Sex == i) \
                    & (dataset.Pclass == j+1),'Age'] = guess_ages[i,j]
                    

    dataset['Age'] = dataset['Age'].astype(int)
    print()
print('Done! \n\n\n')
train_df.head()

<br>

**Splitting age into ordinal bands**

In [ ]:
# Age bands
train_df['AgeBand'] = pd.cut(train_df['Age'], 5)
train_df[['AgeBand', 'Survived']].groupby(['AgeBand'], as_index=False)\
                    .mean().sort_values(by='AgeBand', ascending=True)

<br>

**Distribution of survival relative to age**

In [ ]:
# Plot distributions of Age of passangers who survived 
# or did not survive

plot_distribution(train_df , var = 'Age' , target = 'Survived', row = 'Sex')

# Recall: {'male': 0, 'female': 1}

<br>

**Label encoding age bands (0 to 4)**

Age bands have a natural order (younger to older), so label encoding (integers 0-4) is appropriate here. Unlike unordered categoricals, where dummy encoding avoids implying a false ordinal relationship, age bands genuinely rank from low to high.

In [ ]:
# Change Age column to
# map Age ranges (AgeBands) to integer values using label Encoding

for dataset in combine:    
    dataset.loc[ dataset['Age'] <= 16, 'Age'] = 0
    dataset.loc[(dataset['Age'] > 16) & (dataset['Age'] <= 32), 'Age'] = 1
    dataset.loc[(dataset['Age'] > 32) & (dataset['Age'] <= 48), 'Age'] = 2
    dataset.loc[(dataset['Age'] > 48) & (dataset['Age'] <= 64), 'Age'] = 3
    dataset.loc[ dataset['Age'] > 64, 'Age']=4
train_df.head()

# Note we could just run 
# dataset['Age'] = pd.cut(dataset['Age'], 5,labels=[0,1,2,3,4])

In [ ]:
# remove AgeBand column

train_df = train_df.drop(['AgeBand'], axis=1)
combine = [train_df, test_df]
train_df.head()

___

**Exploration question:** Does traveling party size predict survival? The code below explores this and collapses FamilySize to a simpler binary signal.

___

In [ ]:
# SibSp = Number of Sibling / Spouses
# Parch = Parents / Children

for dataset in combine:
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1

    
# Survival chance against FamilySize

train_df[['FamilySize', 'Survived']].groupby(['FamilySize'], as_index=True) \
                                .mean().sort_values(by='Survived', ascending=False)

In [ ]:
# Plot it, 1 is survived

sns.countplot(x='Survived', hue="FamilySize", data=train_df, order=[1,0]);

In [ ]:
# Create binary variable if the person was alone or not

for dataset in combine:
    dataset['IsAlone'] = 0
    dataset.loc[dataset['FamilySize'] == 1, 'IsAlone'] = 1

train_df[['IsAlone', 'Survived']].groupby(['IsAlone'], as_index=True).mean()

<br>

**Collapsing FamilySize to a binary feature**

`FamilySize` has nine distinct values, many with very few observations. A binary `IsAlone` flag retains the predictive signal while reducing noise.

In [ ]:
# We will only use the binary IsAlone feature for further analysis

for df in combine:
    df.drop(['Parch', 'SibSp', 'FamilySize'], axis=1, inplace=True)


train_df.head()

<br>

**Creating an interaction feature: Age x Pclass**

This interaction term captures the joint effect of being young and poor (high `Age*Class` value) versus old and wealthy (low value). It compresses two columns into one combined signal.

In [ ]:
# We can also create new features based on intuitive combinations
# Here is an example when we say that the age times socioclass is a determinant factor

for dataset in combine:
    dataset['Age*Class'] = dataset.Age * dataset.Pclass

train_df.loc[:, ['Age*Class', 'Age', 'Pclass']].head()

In [ ]:
train_df[['Age*Class', 'Survived']].groupby(['Age*Class'], as_index=True).mean()

___

**Exploration question:** Is the port of embarkation a predictor for survival?

___

___

**Note:** Third Class passengers were the first to board, with First and Second Class following up to an hour before departure. Third Class passengers were inspected for ailments that might lead to their being refused entry to the United States; First Class passengers were personally greeted by Captain Smith.

___

<br>

**Dummy encoding port of embarkation**

Embarked has three categories (C = Cherbourg, Q = Queenstown, S = Southampton) with no natural ordering, so dummy (one-hot) encoding avoids implying a false ordinal relationship.

In [ ]:
train_df.columns

In [ ]:
# To replace Nan value in 'Embarked', we will use the mode
# in 'Embaraked'. This will give us the most frequent port 
# the passengers embarked from

freq_port = train_df['Embarked'].dropna().mode()[0]
print('Most frequent port of Embarkation:',freq_port)


In [ ]:
# Fill NaN 'Embarked' Values in the datasets

for dataset in combine:
    dataset['Embarked'] = dataset['Embarked'].fillna(freq_port)
    
    
train_df[['Embarked', 'Survived']].groupby(['Embarked'], as_index=True) \
                    .mean().sort_values(by='Survived', ascending=False)


<br>

**Survival rate by port of embarkation**

In [ ]:
# Plot of relationship between survival and origin 

sns.countplot(x='Survived', hue="Embarked", data=train_df, order=[1,0])
plt.xticks(range(2),['Made it!', 'Deceased']);


In [ ]:
# Create categorical dummy variables for Embarked values

for dataset in combine:
    binary_encoded = pd.get_dummies(dataset.Embarked)
    print(binary_encoded.head(3))
    newcols = binary_encoded.columns
    dataset[newcols] = binary_encoded

    
train_df.head()

In [ ]:
# Drop Embarked

for dataset in combine:
    dataset.drop('Embarked', axis=1, inplace=True)

___

**Exploration question:** What is the relationship between ticket `Fare` and survival?

___

In [ ]:
# Fill the NA values in the Fares column with the median

test_df['Fare'].fillna(test_df['Fare'].dropna().median(), inplace=True)
test_df.head()

In [ ]:
# q cut will find ranges equal to the quartile of the data

train_df['FareBand'] = pd.qcut(train_df['Fare'], 4)
train_df[['FareBand', 'Survived']].groupby(['FareBand'], as_index=False).mean().sort_values(by='FareBand', ascending=True)

In [ ]:
for dataset in combine:
    dataset['Fare']=pd.qcut(train_df['Fare'],4,labels=np.arange(4))
    dataset['Fare'] = dataset['Fare'].astype(int)

train_df[['Fare','FareBand']].head()

In [ ]:
# Drop FareBand

train_df = train_df.drop(['FareBand'], axis=1) 
combine = [train_df, test_df]

___

**Exploration question:** Does cabin tier predict survival?

___

In [ ]:
train_df.Cabin.unique()

In [ ]:
train_df['CabinType'] = train_df['Cabin'].apply(lambda x: str(x)[0] if pd.notnull(x) else None)

<br>

**Survival by cabin tier**

In [ ]:
sns.countplot(x='Survived',
              hue="CabinType",
              palette="ch:.25",
              data=train_df.sort_values(by='CabinType'),
              order=[1,0]);

In [ ]:
# Missing data?

print(train_df['CabinType'].isnull().sum())
print(len(train_df['CabinType']))

<br>

**Dropping non-predictive columns**

Cabin is over 77% missing; its tier signal is already captured by Fare. Ticket is unique per passenger and carries no generalizable information.

In [ ]:
# drop cabin related features as the predictive power is captured by `fare`
train_df = train_df.drop(['Cabin', 'CabinType'], axis=1)
test_df = test_df.drop(['Cabin'], axis=1)
combine = [train_df, test_df]

In [ ]:
# drop `Ticket` since each ticket is unique and no value in predicting survival

train_df = train_df.drop(['Ticket'], axis=1)
test_df = test_df.drop(['Ticket'], axis=1)
combine = [train_df, test_df]

In [ ]:
# sanity check
train_df.head()

<hr style="border: 1px solid#003262;" />

**Preprocessing complete - sanity checks**

In [ ]:
# All features are approximately on the same scale
# no need for feature engineering / normalization

train_df.head(7)

In [ ]:
test_df.head(7)

<br>

**Pearson correlation between features in the preprocessed dataset**

Uncorrelated features are generally more powerful predictors. Features that are highly correlated with each other add redundancy without adding information.

In [ ]:
# Uncorrelated features are generally more powerful predictors

colormap = plt.cm.viridis
plt.figure(figsize=(12,12))
plt.title('Pearson Correlation of Features', y=1.05, size=15)
sns.heatmap(train_df.corr().round(2)\
            ,linewidths=0.1,vmax=1.0, square=True, cmap=colormap, \
            linecolor='white', annot=True);

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The `Fare` column was binned into four quartile groups (0 = lowest, 3 = highest). Create a binary feature `IsTopFare` that equals 1 when Fare is in the top quartile (value 3) and 0 otherwise. Compute the mean survival rate for each group. Does top-quartile fare strongly predict survival?**

<br>

<hr style="border: 2px solid#003262;" />

In [ ]:
# Create IsTopFare feature (Fare quartile 3 = top 25% of fares)
train_df['IsTopFare'] = (train_df['Fare'] == 3).astype(int)

# Survival rate by top fare vs. everyone else
survival_by_top_fare = train_df.groupby('IsTopFare')['Survived'].mean().reset_index()
survival_by_top_fare

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **STORING** PREPROCESSED DATA IN **SQLITE**

Storing preprocessed data in SQLite simulates a real production pattern: raw data lives in files or object storage, cleaned data lives in a relational store, and downstream consumers query that store. The round-trip query confirms the schema was written correctly.

Storing in SQLite also decouples preprocessing from modeling - a teammate could query the database directly without re-running the preprocessing pipeline.

In [ ]:
import sqlite3

# Establish a SQLite connection
conn = sqlite3.connect('titanic_survivor_analysis.db')

# Save the preprocessed training dataset to the SQLite database
train_df.to_sql('train_data', conn, if_exists='replace', index=False)

# Query to ensure the data was stored correctly
df_from_db = pd.read_sql('SELECT * FROM train_data', conn)
df_from_db.head()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Write a SQL query that computes the average `Age` and average `Fare` for survivors (`Survived = 1`) versus non-survivors (`Survived = 0`). Do survivors tend to be younger or older? Did they pay more or less for their ticket?**

<br>

<hr style="border: 2px solid#003262;" />

In [ ]:
import sqlite3
conn = sqlite3.connect('titanic_survivor_analysis.db')

query = (
    "SELECT "
    "    Survived, "
    "    ROUND(AVG(Age), 2)  AS avg_age, "
    "    ROUND(AVG(Fare), 2) AS avg_fare "
    "FROM train_data "
    "GROUP BY Survived"
)
result = pd.read_sql(query, conn)
result

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_5'></a>

<hr style="border: 2px solid#003262;" />

#### PART 5

## **MODEL BUILDING** AND EVALUATION

We evaluate five classification algorithms on a held-out 20% validation split. All five cluster around 76-84%. The ceiling is roughly 80% because survival for any individual passenger was partly determined by chance - who happened to be near a lifeboat, which deck they were on, which crew members they encountered. See Appendix I for the human story behind this ceiling.

**Algorithms compared:**
1. Logistic Regression
2. K-Nearest Neighbors (KNN)
3. Support Vector Machines (SVM)
4. XGBoost
5. Random Forest

<br>

**Setting up training and validation sets**

We hold out 20% of the labeled training data as a validation set. The models never see this data during training - it is used only to estimate real-world performance.

In [ ]:
X = train_df.drop("Survived", axis=1) # Training & Validation data
Y = train_df["Survived"]              # Response / Target Variable

X_submission  = test_df.drop("PassengerId", axis=1).copy()

print(X.shape, Y.shape)

In [ ]:
# Split training set so that we validate on 20% of the data
# Note that our algorithms will never have seen the validation 
# data during training. This is to evaluate how good our estimators are.

np.random.seed(1337) # set random seed for reproducibility

from sklearn.model_selection import train_test_split

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2)

print(X_train.shape, Y_train.shape)
print(X_val.shape, Y_val.shape)

___

**Note:** The scikit-learn ML workflow for each algorithm is the same three steps:
1. **Instantiate** the model object
2. **Fit** the model to training data
3. **Predict and evaluate** - compare predictions on the held-out validation set to true labels

___

<br>

**Comparing five classifiers**

**1. Logistic Regression**

Models the probability of survival as a sigmoid function of a weighted sum of features. Interpretable and fast; works well when the decision boundary is approximately linear.

In [ ]:
logreg = LogisticRegression()                                # instantiate
logreg.fit(X_train, Y_train)                                 # fit
Y_pred = logreg.predict(X_val)                               # predict
acc_logreg = sum(Y_pred == Y_val)/len(Y_val)*100             # evaluate

print('Logistic Regression labeling accuracy:', str(round(acc_logreg,2)),'%')

In [ ]:
# we could also use scikit learn's method score
# that predicts and then compares to validation set labels
acc_log_2 = logreg.score(X_val, Y_val)                       # evaluate

print('Logistic Regression using built-in method:', str(round(acc_log_2*100,2)),'%')

**2. K-Nearest Neighbors (KNN)**

Classifies each passenger by majority vote among the k most similar passengers in the training set. No explicit model is learned; all computation happens at prediction time.

In [ ]:
knn = KNeighborsClassifier(n_neighbors = 3)                  # instantiate
knn.fit(X_train, Y_train)                                    # fit
acc_knn = knn.score(X_val, Y_val)                            # predict + evaluate

print('K-Nearest Neighbors labeling accuracy:', str(round(acc_knn*100,2)),'%')                                

**3. Support Vector Machines (SVM)**

Finds the hyperplane that maximally separates the two classes in a high-dimensional feature space. Effective in high dimensions and robust to outliers; less interpretable than Logistic Regression.

In [ ]:
# Support Vector Machines Classifier (non-linear kernel)
svc = SVC()                                                  # instantiate
svc.fit(X_train, Y_train)                                    # fit
acc_svc = svc.score(X_val, Y_val)                            # predict + evaluate

print('Support Vector Machines labeling accuracy:', str(round(acc_svc*100,2)),'%')

**4. XGBoost**

An ensemble method that builds decision trees sequentially, each one correcting the errors of the previous. Often the strongest performer on tabular data, especially with careful hyperparameter tuning.

In [ ]:
# XGBoost, same API as scikit-learn
gradboost = xgb.XGBClassifier(n_estimators=1000)             # instantiate
gradboost.fit(X_train, Y_train)                              # fit
acc_xgboost = gradboost.score(X_val, Y_val)                  # predict + evalute

print('XGBoost labeling accuracy:', str(round(acc_xgboost*100,2)),'%')

**5. Random Forest**

Trains many decision trees independently on random subsets of the data and features, then aggregates their votes. Reduces variance compared to a single deep tree and provides feature importance scores.

In [ ]:
# Random Forest
random_forest = RandomForestClassifier(n_estimators=500)    # instantiate
random_forest.fit(X_train, Y_train)                         # fit
acc_rf = random_forest.score(X_val, Y_val)                  # predict + evaluate

print('K-Nearest Neighbors labeling accuracy:', str(round(acc_rf*100,2)),'%')

<br>

**Feature importance scores from the Random Forest**

Random Forest measures importance as the mean decrease in impurity across all trees where a feature is used as a split node. Higher importance means the feature contributes more to reducing prediction error.

In [ ]:
# Look at importnace of features for random forest

def plot_model_var_imp( model , X , y ):
    imp = pd.DataFrame( 
        model.feature_importances_  , 
        columns = [ 'Importance' ] , 
        index = X.columns 
    )
    imp = imp.sort_values( [ 'Importance' ] , ascending = True )
    imp[ : 10 ].plot( kind = 'barh' )
    print ('Training accuracy Random Forest:',model.score( X , y ))

plot_model_var_imp(random_forest, X_train, Y_train)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The KNN model was trained with `k=3`. Re-train it with `k=1`, `k=5`, and `k=15`. Print the validation accuracy for each. As k increases, does the model become more or less flexible? Which k value do you expect to generalize best to unseen data, and why?**

<br>

<hr style="border: 2px solid#003262;" />

In [ ]:
for k in [1, 5, 15]:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train, Y_train)
    acc_k = knn_k.score(X_val, Y_val)
    print(f'KNN (k={k}) validation accuracy: {round(acc_k * 100, 2)}%')

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Appendix_1'></a>

<hr style="border: 2px solid#003262;" />

#### APPENDIX I

## WHY DO MODELS **MAX OUT** AT ~80%?

**John Jacob Astor**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/john-jacob-astor.jpg" align="center" width="30%" padding="10px"><br>
    <br>
    John Jacob Astor IV (1864-1912)
</div>

John Jacob Astor perished in the disaster even though our model predicted he would survive. Astor was the wealthiest person on the Titanic - his ticket fare was valued at over 35,000 USD in 2016 - and it seems likely that he would have been among the approximately 35 percent of men in first class to survive. However, this was not the case: although his pregnant wife survived, John Jacob Astor's body was recovered a week later, along with a gold watch, a diamond ring with three stones, and over 92,000 USD (2016 value) in cash.

<br>

**Olaus Jorgensen Abelseth**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/olaus-jorgensen-abelseth.jpg" align="center" width="30%" padding="10px"><br>
    <br>
    Olaus Jorgensen Abelseth (1887-1980)
</div>

Abelseth was a 25-year-old Norwegian sailor, a man in 3rd class, and not expected to survive by our classifier. Once the ship sank, he survived by swimming for 20 minutes in the frigid North Atlantic water before joining other survivors on a waterlogged collapsible boat. Abelseth married three years later, settled as a farmer in North Dakota, had four children, and died in 1980 at the age of 94.

<br>

**Key takeaway**

As engineers and business professionals, we are trained to ask: what could we do to improve on 80 percent? It is easy to forget that these data points represent real people. Each time our model is wrong, we should be glad - in such misclassifications we often find incredible stories of human nature and courage triumphing over extremely difficult odds.

___

**It is important to never lose sight of the human element when analyzing data that deals with people.** In the case of this dataset, the moment we are disappointed that accuracy was not higher, we are disappointed that more people did not die.

___

<a id='Appendix_2'></a>

<hr style="border: 2px solid#003262;" />

#### APPENDIX II

## RESOURCES AND REFERENCES

**Algorithm references:**

- Gradient Boosting: "A Kaggle Master Explains Gradient Boosting" - Kaggle Blog, 2017
- K-Nearest Neighbors: Towards Data Science, "Introduction to K-Nearest Neighbors"
- Logistic Regression: Towards Data Science, "5 Reasons Logistic Regression Should Be the First Thing You Learn"
- Naive Bayes: scikit-learn documentation - sklearn.naive_bayes
- Random Forest: Will Koehrsen, "Random Forest: Simple Explanation" - Medium
- Support Vector Machines: Towards Data Science, "Support Vector Machines"

**General resources:**

- Kaggle Titanic: Machine Learning from Disaster - competition page and community notebooks
- scikit-learn User Guide: Model evaluation and selection - train/test splits, cross-validation, scoring
- SQLite documentation: sqlite.org
- pandas documentation: pandas.pydata.org
- "An Introduction to Statistical Learning" - regression and classification foundations
- "Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow" - practical scikit-learn workflows

<hr style="border: 6px solid#003262;" />